## 3.5 Drill-Down: Quem Fala, Sobre o Quê, e Com Qual Sentimento

### Motivação

As seções 3.1–3.4 demonstraram que **Tier e Classificação são estatisticamente associados** (χ², resíduos padronizados). Mas o teste qui-quadrado responde *se* há associação — não *quem* a produz nem *sobre o quê*.

Esta seção abre a caixa preta com três perguntas operacionais:

1. **Quais veículos e programas** de alta relevância geram mais opinião (positiva ou negativa)?
2. **Quais categorias e subcategorias** concentram a negatividade nos veículos grandes?
3. **Existe divergência temática entre Tiers?** (i.e., veículos grandes e pequenos falam dos mesmos assuntos com sentimentos opostos?)

### Critério de volume mínimo

Para evitar que veículos com 2–3 menções distorçam rankings (NSS = −100 com N = 2 não é informativo), aplicaremos um filtro de **N ≥ 30** para veículos e programas, e **N ≥ 20** para subcategorias.

---

### 3.5.1 Perfil de Sentimento dos Veículos por Tier

In [ ]:
def build_vehicle_sentiment_profile(df_input: pd.DataFrame, 
                                     tier: str, 
                                     top_n: int = 15, 
                                     min_mentions: int = 30) -> pd.DataFrame:
    """
    Constrói o perfil de sentimento dos Top N veículos dentro de um Tier específico.
    
    Retorna DataFrame com: Total, Positiva, Neutra, Negativa, %Pos, %Neg, NSS.
    Filtro de volume mínimo evita distorções por amostras pequenas.
    """
    df_tier = df_input[df_input['Tier'] == tier].copy()
    
    profile = df_tier.groupby('Veículo').agg(
        Total=('Classificação', 'count'),
        Positiva=('Classificação', lambda x: (x == 'POSITIVA').sum()),
        Neutra=('Classificação', lambda x: (x == 'NEUTRA').sum()),
        Negativa=('Classificação', lambda x: (x == 'NEGATIVA').sum()),
    )
    
    # Filtro de volume mínimo — decisão metodológica, não estética
    profile = profile[profile['Total'] >= min_mentions]
    
    profile['%Pos'] = (profile['Positiva'] / profile['Total'] * 100).round(1)
    profile['%Neg'] = (profile['Negativa'] / profile['Total'] * 100).round(1)
    profile['NSS'] = ((profile['Positiva'] - profile['Negativa']) / profile['Total'] * 100).round(1)
    
    return profile.sort_values('Total', ascending=False).head(top_n)


# Aplicar para Muito Relevante
profile_mr = build_vehicle_sentiment_profile(df_sentiment, 'Muito Relevante')

print("=" * 100)
print("TOP 15 VEÍCULOS — TIER 'MUITO RELEVANTE' — PERFIL DE SENTIMENTO")
print("=" * 100)
print(profile_mr.to_string())
print("=" * 100)

# Estatísticas resumidas
n_positivos = (profile_mr['NSS'] > 0).sum()
n_negativos = (profile_mr['NSS'] < 0).sum()
nss_mediano = profile_mr['NSS'].median()

print(f"\n📊 Resumo: {n_positivos} veículos com NSS > 0 | {n_negativos} com NSS < 0")
print(f"   NSS mediano do grupo: {nss_mediano:+.1f}")

In [ ]:
def plot_vehicle_nss_divergente(profile_df: pd.DataFrame, tier_label: str) -> None:
    """
    Gráfico de barras divergentes (lollipop horizontal) para NSS por veículo.
    
    Barras à direita (verde) = sentimento líquido positivo.
    Barras à esquerda (vermelho) = sentimento líquido negativo.
    O tamanho do marcador reflete o volume total de menções.
    
    Por que divergente e não barras simples?
    Porque o ponto de referência é o zero (equilíbrio entre pos/neg).
    A distância da barra ao zero é o que importa, e a direção indica polaridade.
    """
    
    df_plot = profile_df.reset_index().sort_values('NSS', ascending=True)
    
    # Cores condicionais ao sinal do NSS
    colors = ['#e74c3c' if nss < 0 else '#2ecc71' for nss in df_plot['NSS']]
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=df_plot['Veículo'],
        x=df_plot['NSS'],
        orientation='h',
        marker_color=colors,
        text=df_plot.apply(lambda r: f"NSS: {r['NSS']:+.1f} (N={r['Total']})", axis=1),
        textposition='outside',
        hovertemplate=(
            "<b>%{y}</b><br>"
            "NSS: %{x:+.1f}<br>"
            "<extra></extra>"
        )
    ))
    
    # Linha de referência no zero
    fig.add_vline(x=0, line_width=2, line_color='#2c3e50', line_dash='solid')
    
    fig.update_layout(
        title=f'<b>NSS por Veículo — Tier: {tier_label}</b><br>'
              f'<sup>Barras verdes = sentimento positivo líquido | Vermelhas = negativo líquido</sup>',
        template='plotly_white',
        height=max(500, len(df_plot) * 40),
        xaxis_title='Net Sentiment Score (NSS)',
        yaxis_title='',
        margin=dict(l=250, r=100, t=80, b=50),
        showlegend=False,
        font=dict(family='Inter, Arial', size=12)
    )
    
    fig.show()


plot_vehicle_nss_divergente(profile_mr, 'Muito Relevante')

#### Interpretação — Veículos de Alta Relevância

O gráfico divergente revela uma **polarização clara** dentro do próprio Tier "Muito Relevante":

**Bloco negativo** (NSS < 0): Veículos jornalísticos com cobertura operacional — TV Globo RJ, Meia Hora, Band News, CBN Rio, RBS TV. São veículos que noticiam **falhas de serviço** (falta de água, rompimentos, extravasamento). O sentimento negativo aqui não é viés editorial — é reflexo da pauta: quando esses veículos cobrem saneamento, é porque algo deu errado.

**Bloco positivo** (NSS > 0): Veículos com pauta institucional/financeira — Valor Econômico (NSS ≈ +78), TV A Crítica, TV Clube, GZH. Cobrem expansão de rede, resultados financeiros, projetos sociais.

**Implicação**: Tratar todos os veículos "Muito Relevante" como um bloco homogêneo é uma simplificação. O NSS Ponderado agrega Valor Econômico e Meia Hora no mesmo peso — mas suas pautas (e portanto seus sentimentos) são estruturalmente diferentes.

---

### 3.5.2 Programas com Maior Polarização

In [ ]:
def build_program_sentiment_profile(df_input: pd.DataFrame, 
                                     top_n: int = 10, 
                                     min_mentions: int = 30,
                                     exclude_generic: list = None) -> pd.DataFrame:
    """
    Perfil de sentimento por Programa, com Tier dominante.
    
    Captura os DOIS extremos: os top_n mais negativos E os top_n mais positivos.
    Sem isso, o gráfico divergente só mostraria um lado — omitindo metade da história.
    
    Exclui programas genéricos (ex: 'Geral') que são catch-all do fornecedor
    de clipping e não representam programas jornalísticos reais.
    """
    if exclude_generic is None:
        exclude_generic = ['Geral']
    
    df_filtered = df_input[~df_input['Programa'].isin(exclude_generic)].copy()
    
    profile = df_filtered.groupby('Programa').agg(
        Total=('Classificação', 'count'),
        Positiva=('Classificação', lambda x: (x == 'POSITIVA').sum()),
        Neutra=('Classificação', lambda x: (x == 'NEUTRA').sum()),
        Negativa=('Classificação', lambda x: (x == 'NEGATIVA').sum()),
        # Moda do Tier — indica de qual nível de relevância vem a maioria das menções
        Tier_Dominante=('Tier', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A'),
    )
    
    profile = profile[profile['Total'] >= min_mentions]
    
    profile['%Pos'] = (profile['Positiva'] / profile['Total'] * 100).round(1)
    profile['%Neg'] = (profile['Negativa'] / profile['Total'] * 100).round(1)
    profile['%Neu'] = (profile['Neutra'] / profile['Total'] * 100).round(1)
    profile['NSS'] = ((profile['Positiva'] - profile['Negativa']) / profile['Total'] * 100).round(1)
    
    # Capturar ambos os extremos e remover duplicatas (caso algum programa apareça nos dois)
    most_negative = profile.nsmallest(top_n, 'NSS')
    most_positive = profile.nlargest(top_n, 'NSS')
    combined = pd.concat([most_negative, most_positive]).drop_duplicates()
    
    return combined.sort_values('NSS', ascending=True)


profile_progs = build_program_sentiment_profile(df_sentiment)

print("=" * 100)
print(f"TOP PROGRAMAS (excl. 'Geral') — {len(profile_progs)} EXTREMOS (mais negativos + mais positivos)")
print("=" * 100)
print(profile_progs[['Total', 'Positiva', 'Negativa', '%Pos', '%Neg', 'NSS', 'Tier_Dominante']].to_string())
print("=" * 100)

In [ ]:
def plot_program_piramide_sentimento(profile_df: pd.DataFrame) -> None:
    """
    Pirâmide de sentimento estilo pirâmide etária.
    
    Eixo Y = Programas (ordenados por NSS, mais negativos embaixo).
    Esquerda = % de menções negativas (vermelho).
    Direita = % de menções positivas (verde).
    Centro/gap = % de neutras (implícito).
    
    Usar % (não absolutos) é a mesma lógica da pirâmide etária real:
    normaliza populações de tamanhos diferentes, tornando-as comparáveis.
    O volume absoluto (N) vai como anotação para dar contexto de impacto.
    """
    
    tier_markers = {
        'Muito Relevante': '🔴',
        'Relevante': '🟠',
        'Menos Relevante': '🔵',
        'Null': '⚪'
    }
    
    df_plot = profile_df.reset_index().sort_values('NSS', ascending=True)
    
    # Labels com Tier e N
    labels = df_plot.apply(
        lambda r: f"{tier_markers.get(r['Tier_Dominante'], '⚪')} {r['Programa']} (N={int(r['Total']):,})", axis=1
    ).tolist()
    
    fig = go.Figure()
    
    # Barras NEGATIVAS — vão para a esquerda (valores negativos no eixo x)
    fig.add_trace(go.Bar(
        y=labels,
        x=-df_plot['%Neg'].values,
        orientation='h',
        name='% Negativa',
        marker_color='#e74c3c',
        marker_opacity=0.85,
        text=df_plot['%Neg'].apply(lambda v: f'{v:.0f}%' if v >= 3 else ''),
        textposition='inside',
        textfont=dict(color='white', size=11),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Negativas: %{customdata[0]:.0f} (%{customdata[1]:.1f}%)<br>'
            'NSS: %{customdata[2]:+.1f}<br>'
            '<extra></extra>'
        ),
        customdata=df_plot[['Negativa', '%Neg', 'NSS']].values,
    ))
    
    # Barras NEUTRAS — também à esquerda, empilhadas após as negativas
    fig.add_trace(go.Bar(
        y=labels,
        x=-df_plot['%Neu'].values,
        orientation='h',
        name='% Neutra',
        marker_color='#bdc3c7',
        marker_opacity=0.6,
        text=df_plot['%Neu'].apply(lambda v: f'{v:.0f}%' if v >= 8 else ''),
        textposition='inside',
        textfont=dict(color='#2c3e50', size=10),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Neutras: %{customdata[0]:.0f} (%{customdata[1]:.1f}%)<br>'
            '<extra></extra>'
        ),
        customdata=df_plot[['Neutra', '%Neu']].values,
    ))
    
    # Barras POSITIVAS — vão para a direita (valores positivos)
    fig.add_trace(go.Bar(
        y=labels,
        x=df_plot['%Pos'].values,
        orientation='h',
        name='% Positiva',
        marker_color='#2ecc71',
        marker_opacity=0.85,
        text=df_plot['%Pos'].apply(lambda v: f'{v:.0f}%' if v >= 3 else ''),
        textposition='inside',
        textfont=dict(color='white', size=11),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Positivas: %{customdata[0]:.0f} (%{customdata[1]:.1f}%)<br>'
            'NSS: %{customdata[2]:+.1f}<br>'
            '<extra></extra>'
        ),
        customdata=df_plot[['Positiva', '%Pos', 'NSS']].values,
    ))
    
    # Linha central (zero)
    fig.add_vline(x=0, line_width=2, line_color='#2c3e50')
    
    fig.update_layout(
        title='<b>Pirâmide de Sentimento por Programa</b><br>'
              '<sup>Esquerda = % Negativa + Neutra | Direita = % Positiva | '
              'Emojis = Tier (🔴MR 🟠Rel 🔵LR ⚪Null)</sup>',
        template='plotly_white',
        height=max(650, len(df_plot) * 38),
        barmode='relative',
        xaxis=dict(
            title='Distribuição Percentual',
            range=[-105, 105],
            tickvals=[-100, -75, -50, -25, 0, 25, 50, 75, 100],
            ticktext=['100%', '75%', '50%', '25%', '0', '25%', '50%', '75%', '100%'],
            zeroline=True,
        ),
        yaxis=dict(title=''),
        margin=dict(l=350, r=30, t=100, b=60),
        legend=dict(
            orientation='h', y=-0.08, x=0.2,
            bgcolor='rgba(255,255,255,0.8)'
        ),
        font=dict(family='Inter, Arial', size=12),
    )
    
    fig.show()


plot_program_piramide_sentimento(profile_progs)

#### Interpretação — Pirâmide de Sentimento

A pirâmide revela o que o NSS sozinho escondia: a **composição** do sentimento de cada programa.

**O que a pirâmide mostra que o NSS não mostra:**
- **Proporção de neutras**: Programas como "Canal Debates" e "Tá na Hora Rio" têm 25–32% de neutras — são mais equilibrados do que o NSS sugere. Programas positivos como "D Santa" e "Rotativo" quase não têm neutras — é positividade pura, o que reforça a hipótese de conteúdo institucional.
- **Volume como contexto**: O (N=...) no label mostra que "Rotativo" (N=3.324) tem 50× mais impacto em volume que "Bom Dia Favela" (N=34), mesmo com NSS parecido. A pirâmide percentual torna os programas visualmente comparáveis, mas o N dá a escala de importância real.

**Padrões por Tier** (emojis no eixo Y):
- 🔴 **Muito Relevante**: Aparece em ambos os extremos — "Tá na Hora Rio" (negativo) e "Revista Canal"/"Bom Dia Favela" (positivos). Confirma que o Tier não determina sentimento; a pauta sim.
- 🔵 **Menos Relevante**: Contém os dois extremos mais radicais — quase 100% negativo ("Boca no Trombone") E quase 100% positivo ("D Santa"). Bipolaridade total.
- ⚪ **Null**: Concentra-se no lado negativo, reforçando o achado dos resíduos na Seção 3.3.

**Ponto de atenção sobre o "Geral"**: Excluído por ser catch-all do fornecedor (42.620 registros — 61% do dataset). Se esses registros tiverem perfil de sentimento diferente dos identificados, há viés de seleção.

---

### 3.5.3 Heatmap: NSS por Categoria × Tier

Esta é a análise central da seção. A pergunta:

**Veículos grandes e pequenos falam dos mesmos temas — mas com sentimentos opostos?**

Se sim, a divergência de NSS entre Tiers não é editorial ("a mídia grande é contra nós"), mas **temática** ("quando a mídia grande cobre saneamento, é porque houve falha operacional").

---

In [ ]:
def build_nss_category_tier_matrix(df_input: pd.DataFrame, 
                                    min_n: int = 30) -> tuple:
    """
    Constrói a matriz NSS (Categoria × Tier) e a matriz de volume (N).
    
    Retorna:
        nss_matrix: DataFrame com NSS por Categoria × Tier
        n_matrix: DataFrame com N por Categoria × Tier
        
    Células com N < min_n recebem NaN (insuficiente para estimativa confiável).
    """
    
    # Ordem dos tiers: do mais ao menos relevante
    tier_order = ['Muito Relevante', 'Relevante', 'Menos Relevante', 'Null']
    
    results = []
    
    for cat in df_input['Categoria'].unique():
        for tier in tier_order:
            subset = df_input[(df_input['Categoria'] == cat) & (df_input['Tier'] == tier)]
            n = len(subset)
            if n >= min_n:
                pos = (subset['Classificação'] == 'POSITIVA').sum()
                neg = (subset['Classificação'] == 'NEGATIVA').sum()
                nss = ((pos - neg) / n) * 100
            else:
                nss = np.nan
            results.append({'Categoria': cat, 'Tier': tier, 'NSS': nss, 'N': n})
    
    df_results = pd.DataFrame(results)
    
    nss_matrix = df_results.pivot(index='Categoria', columns='Tier', values='NSS')[tier_order]
    n_matrix = df_results.pivot(index='Categoria', columns='Tier', values='N')[tier_order]
    
    # Ordenar categorias pelo volume total (as mais relevantes primeiro)
    cat_order = df_input['Categoria'].value_counts().index
    nss_matrix = nss_matrix.reindex(cat_order)
    n_matrix = n_matrix.reindex(cat_order)
    
    return nss_matrix, n_matrix


nss_cat_tier, n_cat_tier = build_nss_category_tier_matrix(df_sentiment)

print("=" * 80)
print("MATRIZ NSS: CATEGORIA × TIER")
print("(NaN = N < 30, insuficiente para estimativa confiável)")
print("=" * 80)
print(nss_cat_tier.round(1).to_string())
print("\n")
print("=" * 80)
print("MATRIZ DE VOLUME (N): CATEGORIA × TIER")
print("=" * 80)
print(n_cat_tier.to_string())

In [ ]:
def plot_heatmap_nss_categoria_tier(nss_matrix: pd.DataFrame, 
                                     n_matrix: pd.DataFrame) -> None:
    """
    Heatmap anotado com NSS por Categoria × Tier.
    
    Escala de cor divergente: vermelho (negativo) → branco (neutro) → verde (positivo).
    Anotação dupla: NSS na linha principal, (N=...) na linha secundária.
    Isso garante que o leitor não interprete um NSS de +95 em N=30 da mesma forma
    que um NSS de +40 em N=6.000.
    """
    
    # Anotações: NSS + volume
    annotations = []
    for i, cat in enumerate(nss_matrix.index):
        for j, tier in enumerate(nss_matrix.columns):
            nss_val = nss_matrix.iloc[i, j]
            n_val = n_matrix.iloc[i, j]
            if pd.notna(nss_val):
                text = f"{nss_val:+.1f}<br><sub>N={int(n_val):,}</sub>"
            else:
                text = "N < 30"
            annotations.append(
                dict(x=tier, y=cat, text=text, showarrow=False,
                     font=dict(size=11, color='#1a202c' if pd.isna(nss_val) or abs(nss_val) < 60 else 'white'))
            )
    
    fig = go.Figure(data=go.Heatmap(
        z=nss_matrix.values,
        x=nss_matrix.columns.tolist(),
        y=nss_matrix.index.tolist(),
        colorscale=[
            [0.0, '#c0392b'],   # negativo forte
            [0.3, '#e74c3c'],   # negativo moderado
            [0.5, '#fdfefe'],   # neutro
            [0.7, '#82e0aa'],   # positivo moderado
            [1.0, '#1e8449'],   # positivo forte
        ],
        zmid=0,  # Centraliza a escala no zero
        colorbar=dict(title='NSS', ticksuffix=''),
        hovertemplate='<b>%{y}</b><br>Tier: %{x}<br>NSS: %{z:+.1f}<extra></extra>',
    ))
    
    fig.update_layout(
        title='<b>Heatmap: NSS por Categoria × Tier</b><br>'
              '<sup>Vermelho = sentimento negativo | Verde = positivo | Anotação inclui volume (N)</sup>',
        template='plotly_white',
        height=max(600, len(nss_matrix) * 45),
        xaxis_title='Tier',
        yaxis_title='',
        yaxis=dict(autorange='reversed'),  # Categorias mais volumosas no topo
        margin=dict(l=200, r=50, t=80, b=50),
        annotations=annotations,
        font=dict(family='Inter, Arial', size=12)
    )
    
    fig.show()


plot_heatmap_nss_categoria_tier(nss_cat_tier, n_cat_tier)

In [ ]:
# Cálculo da divergência entre Tiers por Categoria
# A divergência é a diferença absoluta de NSS entre Muito Relevante e Menos Relevante.
# Quanto maior, mais o sentimento muda conforme o Tier — evidência de cobertura temática diferenciada.

divergencia = (nss_cat_tier['Menos Relevante'] - nss_cat_tier['Muito Relevante']).dropna().sort_values(ascending=False)

print("=" * 80)
print("DIVERGÊNCIA DE NSS: (MENOS RELEVANTE) − (MUITO RELEVANTE)")
print("Valores positivos = veículos pequenos mais otimistas que grandes")
print("=" * 80)
for cat, div in divergencia.items():
    mr_val = nss_cat_tier.loc[cat, 'Muito Relevante']
    lr_val = nss_cat_tier.loc[cat, 'Menos Relevante']
    print(f"  {cat:35s}: Δ = {div:+6.1f}  (MR: {mr_val:+6.1f} → LR: {lr_val:+6.1f})")

#### Interpretação — Heatmap Categoria × Tier

Este é possivelmente o achado mais importante desta seção. O heatmap revela que a divergência entre Tiers **não é uniforme** — ela se concentra em categorias operacionais:

**Maior divergência** (Δ > 50 pontos de NSS entre Muito Relevante e Menos Relevante):
- **Qualidade da Água**: Δ ≈ 92 pontos — a maior divergência de todo o dataset. Nos veículos grandes, é pauta de crise (água com coloração, contaminação). Nos pequenos, é comunicado sobre tratamento.
- **Esgoto**: Δ ≈ 89 pontos — veículos grandes cobrem extravasamento; pequenos, expansão da rede.
- **Abastecimento**: Δ ≈ 82 pontos. Quando um veículo grande fala de abastecimento, é falta de água. Quando um pequeno fala, é comunicado institucional sobre obras.
- **Serviços**: Δ ≈ 57 pontos — padrão similar, menos extremo.

**Sem divergência** (Δ < 10 pontos):
- **Projetos Sociais**: NSS ≈ +95 em todos os Tiers. Quando o tema é social, o sentimento é positivo independentemente da relevância do veículo.
- **Holding**: Cobertura financeira/institucional, uniformemente positiva.

**Implicação estratégica**: A negatividade nos veículos grandes não é um problema de *relações com a mídia* — é um problema de *operação*. Melhorar o NSS Ponderado exige reduzir falhas de abastecimento e esgoto, não melhorar a comunicação com jornalistas.

---

### 3.5.4 Subcategorias: Onde Está a Negatividade e Onde Está a Força

O drill-down final: dentro do Tier alto, **quais subcategorias específicas** concentram negatividade e quais concentram positividade?

Mostrar apenas o lado negativo daria uma visão distorcida — o Tier Muito Relevante tem 80 subcategorias com NSS positivo contra 32 negativas. A diferença é que as negativas são poucas mas **volumosas** (falta de água, vazamento), enquanto as positivas são mais dispersas.

---

In [ ]:
def build_subcategory_profile_tier_alto(df_input: pd.DataFrame, 
                                         tier: str = 'Muito Relevante',
                                         top_n: int = 10,
                                         min_mentions: int = 20) -> pd.DataFrame:
    """
    Perfil de sentimento das subcategorias no Tier especificado.
    
    Captura os dois extremos: as top_n mais negativas E as top_n mais positivas.
    Ordenação final por NSS para visualizar o espectro completo.
    """
    df_tier = df_input[df_input['Tier'] == tier].copy()
    
    profile = df_tier.groupby(['Categoria', 'Subcategoria']).agg(
        Total=('Classificação', 'count'),
        Positiva=('Classificação', lambda x: (x == 'POSITIVA').sum()),
        Negativa=('Classificação', lambda x: (x == 'NEGATIVA').sum()),
    )
    
    profile = profile[profile['Total'] >= min_mentions]
    
    profile['%Pos'] = (profile['Positiva'] / profile['Total'] * 100).round(1)
    profile['%Neg'] = (profile['Negativa'] / profile['Total'] * 100).round(1)
    profile['NSS'] = ((profile['Positiva'] - profile['Negativa']) / profile['Total'] * 100).round(1)
    
    # Capturar ambos os extremos
    most_negative = profile.nsmallest(top_n, 'NSS')
    most_positive = profile.nlargest(top_n, 'NSS')
    combined = pd.concat([most_negative, most_positive]).drop_duplicates()
    
    return combined.sort_values('NSS', ascending=True)


subcats_mr = build_subcategory_profile_tier_alto(df_sentiment)

print("=" * 100)
print("SUBCATEGORIAS EXTREMAS — TIER 'MUITO RELEVANTE' — 10 MAIS NEGATIVAS + 10 MAIS POSITIVAS")
print("(Filtro: N ≥ 20)")
print("=" * 100)
print(subcats_mr.to_string())
print("=" * 100)

# Concentração negativa
total_neg_mr = df_sentiment[df_sentiment['Tier'] == 'Muito Relevante']['Classificação'].eq('NEGATIVA').sum()
top5_neg = subcats_mr.nsmallest(5, 'NSS')['Negativa'].sum()
print(f"\n📊 Concentração negativa: as 5 subcategorias mais negativas explicam "
      f"{top5_neg / total_neg_mr * 100:.1f}% de todas as menções negativas no Tier Muito Relevante.")

# Concentração positiva
total_pos_mr = df_sentiment[df_sentiment['Tier'] == 'Muito Relevante']['Classificação'].eq('POSITIVA').sum()
top5_pos = subcats_mr.nlargest(5, 'NSS')['Positiva'].sum()
print(f"📊 Concentração positiva: as 5 subcategorias mais positivas explicam "
      f"{top5_pos / total_pos_mr * 100:.1f}% de todas as menções positivas no Tier Muito Relevante.")

In [ ]:
def plot_treemap_tier_categoria(df_input: pd.DataFrame, 
                                  min_mentions: int = 10) -> None:
    """
    Treemap com hierarquia Tier > Categoria.
    
    Mudança de design em relação à versão anterior:
    - Antes: filtrava UM Tier e mostrava Categoria > Subcategoria.
    - Agora: mostra TODOS os Tiers como nível superior, com Categorias dentro.
    
    Por quê? Porque a pergunta central da Seção 3 é 'como o mesmo tema muda
    de sentimento conforme o Tier'. A hierarquia Tier > Categoria permite VER
    essa divergência: 'Abastecimento' aparece vermelho em Muito Relevante
    e verde em Menos Relevante — lado a lado.
    
    Tamanho = volume total | Cor = NSS (divergente vermelho → verde)
    
    Subcategoria fica no hover (3 níveis no treemap ficaria ilegível com 559 blocos).
    """
    
    treemap_data = df_input.groupby(['Tier', 'Categoria']).agg(
        Total=('Classificação', 'count'),
        Positiva=('Classificação', lambda x: (x == 'POSITIVA').sum()),
        Negativa=('Classificação', lambda x: (x == 'NEGATIVA').sum()),
    ).reset_index()
    
    treemap_data = treemap_data[treemap_data['Total'] >= min_mentions]
    treemap_data['NSS'] = ((treemap_data['Positiva'] - treemap_data['Negativa']) / treemap_data['Total'] * 100).round(1)
    
    # Ordenar Tiers logicamente (não alfabeticamente)
    tier_order = {'Muito Relevante': 1, 'Relevante': 2, 'Menos Relevante': 3, 'Null': 4}
    treemap_data['Tier_Order'] = treemap_data['Tier'].map(tier_order)
    treemap_data = treemap_data.sort_values(['Tier_Order', 'Total'], ascending=[True, False])
    
    fig = px.treemap(
        treemap_data,
        path=['Tier', 'Categoria'],
        values='Total',
        color='NSS',
        color_continuous_scale=[
            [0.0, '#c0392b'],   # negativo forte
            [0.3, '#e74c3c'],   # negativo moderado
            [0.5, '#fdfefe'],   # neutro (zero)
            [0.7, '#82e0aa'],   # positivo moderado
            [1.0, '#1e8449'],   # positivo forte
        ],
        color_continuous_midpoint=0,
        title='<b>Mapa de Sentimento: Tier × Categoria</b><br>'
              '<sup>Tamanho = volume | Cor = NSS | Compare a mesma Categoria entre Tiers</sup>',
    )
    
    fig.update_layout(
        template='plotly_white',
        height=700,
        margin=dict(l=10, r=10, t=80, b=10),
        font=dict(family='Inter, Arial', size=12),
        coloraxis_colorbar=dict(title='NSS', ticksuffix=''),
    )
    
    fig.update_traces(
        hovertemplate=(
            '<b>%{label}</b><br>'
            'Total: %{value:,} menções<br>'
            'NSS: %{color:+.1f}<br>'
            '<extra></extra>'
        ),
        textinfo='label+value'
    )
    
    fig.show()


plot_treemap_tier_categoria(df_sentiment)

#### Interpretação — Treemap Tier × Categoria

O treemap com Tier como nível superior transforma a tabela de divergência (Seção 3.5.3) em imagem. O que antes eram números, agora é visual:

**O que salta aos olhos:**
- **Abastecimento** é o maior bloco em quase todos os Tiers — mas muda de cor. No Muito Relevante é vermelho (NSS ≈ −42), no Menos Relevante é verde (NSS ≈ +40). Mesma categoria, sentimentos opostos conforme a relevância do veículo.
- **Projetos Sociais** e **Obras e Melhorias** são verdes em todos os Tiers. Quando o tema é positivo por natureza, o Tier não muda o sentimento — é cobertura favorável independente do tamanho do veículo.
- **Comunicados** fica branco/neutro (NSS próximo de zero) em todos os Tiers. Faz sentido: comunicados são informativos, não opinativos.
- O bloco **Null** (veículos sem Tier classificado) tem proporções similares ao Muito Relevante em volume, mas com Abastecimento tendendo ao branco/neutro — reforçando que esses veículos não classificados têm perfil próprio.

**Leitura comparativa (clique nos blocos de Tier para expandir):**
- O Tier "Menos Relevante" é o maior de todos (33.387 menções) e majoritariamente verde. É o Tier que mais puxa o NSS Simples para cima.
- O Tier "Muito Relevante" é o menor (9.482 menções) mas com mais vermelho. É o Tier que puxa o NSS Ponderado para baixo.

**O insight para stakeholders**: Se alguém perguntar "por que o NSS Ponderado é menor que o Simples?", este treemap é a resposta visual — o vermelho se concentra onde o peso é maior.

---

### 3.5.5 Síntese da Seção 3.5

| Achado | Evidência | Implicação |
|--------|-----------|------------|
| Veículos Muito Relevante são polarizados internamente | NSS varia de −39 (Meia Hora) a +78 (Valor Econômico) | Tratar Tier alto como bloco homogêneo é simplificação perigosa |
| Tier Menos Relevante também contém extremos negativos | Programas de denúncia popular ("Boca no Trombone", "Fala Baixada") com NSS < −85 | A equação "veículo pequeno = positivo" é simplificação; segmentar por tipo de pauta é mais preciso |
| A divergência de NSS entre Tiers concentra-se em Abastecimento, Esgoto e Qualidade da Água | Δ ≈ 82 pts em Abastecimento, 89 em Esgoto, 92 em Qualidade da Água | A negatividade não é editorial — é reflexo operacional |
| Negatividade no Tier alto é concentrada em poucas subcategorias | Top 5 subcategorias explicam ~67% das menções negativas em MR | Ações corretivas focadas (falta de água, vazamento, rompimento) podem ter impacto desproporcional no NSS Ponderado |

---